# Diferential expression analysis by Sex

In [1]:
import scanpy as sc
import decoupler as dc

# Only needed for processing
import numpy as np
import pandas as pd
from anndata import AnnData
from pydeseq2.dds import DeseqDataSet, DefaultInference
from pydeseq2.ds import DeseqStats


In [2]:
experiment = 'RNAseq_abundances_adjusted_combat_inmose'
age = 'young'
comparison = 'male.vs.female'

In [3]:
adata = pd.read_csv(f'/home/amore/work/data/{experiment}_gene_symbol_expression.csv', index_col=0)
adata

,TSPAN6,TNMD,DPM1,SCYL3,FIRRM,FGR,CFH,FUCA2,GCLC,NFYA,...,C4orf36.1,TUSC2P1,Unnamed: 34321,OR4M2-OT1,H2BK1,OR1Q1BP,Unnamed: 34325,Unnamed: 34326,TBCEL-TECTA,Age
Sample,,,,,,,,,,,,,,,,,,,,,
SRR13758984,21281762,0,15759712,0,11295483,0,35294310,0,18333093,0,...,12741169,0,0,0,0,0,0,2773,0,91.0
SRR13758985,8542646,0,15035432,0,0,0,26637602,0,1389125,15716930,...,10663212,0,0,0,0,0,0,1328,0,86.0
SRR13758986,4079945,0,5410942,0,5658191,0,10946301,4491317,7747412,9046855,...,4248288,0,0,0,0,0,0,1,0,69.0
SRR13758987,14758557,0,12809471,0,0,0,24093106,0,11562944,16086208,...,10497182,0,0,0,0,0,0,1614,0,83.0
SRR13758988,3623061,0,15529838,23957067,0,0,25492722,10773730,12889518,0,...,10506926,0,0,0,0,0,0,795,0,71.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SRR1555210,37402439,0,21355420,1,1212397,1,87213838,3370216,5884457,951798,...,23486240,0,0,0,0,0,0,0,5351828,27.5
SRR1555211,48027788,0,89024445,1348907,24399,1,148387777,3685378,113719675,2233983,...,34582470,0,0,0,1956002,0,0,0,0,27.5
SRR1555212,84899250,0,167383416,407713,37284600,1,219777948,58357903,79405469,147215871,...,81916604,0,1537700,0,824055,0,1040940,12712840,5952181,27.5


In [4]:
metadata_file = '/home/amore/work/data/All_rna_samples_metadata.csv'
metadata_df = pd.read_csv(metadata_file, index_col=0)
metadata_df

,Age,Status,Experiment,Sex
Run,,,,
SRR13758984,91.0,Sarcopenia,GSE167186,NaN
SRR13758985,86.0,Healthy,GSE167186,male
SRR13758986,69.0,Healthy,GSE167186,male
SRR13758987,83.0,Sarcopenia,GSE167186,NaN
SRR13758988,71.0,UNCLASSIFIED,GSE167186,NaN
...,...,...,...,...
SRR1555214,27.5,untrained,GSE60590,male
SRR1555215,27.5,untrained,GSE60590,male
SRR1555216,27.5,trained,GSE60590,male


In [5]:
metadata_df.drop(columns=['Age','Status','Experiment'], inplace=True)
metadata_df

,Sex
Run,
SRR13758984,NaN
SRR13758985,male
SRR13758986,male
SRR13758987,NaN
SRR13758988,NaN
...,...
SRR1555214,male
SRR1555215,male
SRR1555216,male


In [6]:
adata.head(2)

,TSPAN6,TNMD,DPM1,SCYL3,FIRRM,FGR,CFH,FUCA2,GCLC,NFYA,...,C4orf36.1,TUSC2P1,Unnamed: 34321,OR4M2-OT1,H2BK1,OR1Q1BP,Unnamed: 34325,Unnamed: 34326,TBCEL-TECTA,Age
Sample,,,,,,,,,,,,,,,,,,,,,
SRR13758984,21281762,0,15759712,0,11295483,0,35294310,0,18333093,0,...,12741169,0,0,0,0,0,0,2773,0,91.0
SRR13758985,8542646,0,15035432,0,0,0,26637602,0,1389125,15716930,...,10663212,0,0,0,0,0,0,1328,0,86.0


In [7]:
adata['Sex'] = metadata_df['Sex']
adata

,TSPAN6,TNMD,DPM1,SCYL3,FIRRM,FGR,CFH,FUCA2,GCLC,NFYA,...,TUSC2P1,Unnamed: 34321,OR4M2-OT1,H2BK1,OR1Q1BP,Unnamed: 34325,Unnamed: 34326,TBCEL-TECTA,Age,Sex
Sample,,,,,,,,,,,,,,,,,,,,,
SRR13758984,21281762,0,15759712,0,11295483,0,35294310,0,18333093,0,...,0,0,0,0,0,0,2773,0,91.0,NaN
SRR13758985,8542646,0,15035432,0,0,0,26637602,0,1389125,15716930,...,0,0,0,0,0,0,1328,0,86.0,male
SRR13758986,4079945,0,5410942,0,5658191,0,10946301,4491317,7747412,9046855,...,0,0,0,0,0,0,1,0,69.0,male
SRR13758987,14758557,0,12809471,0,0,0,24093106,0,11562944,16086208,...,0,0,0,0,0,0,1614,0,83.0,NaN
SRR13758988,3623061,0,15529838,23957067,0,0,25492722,10773730,12889518,0,...,0,0,0,0,0,0,795,0,71.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SRR1555210,37402439,0,21355420,1,1212397,1,87213838,3370216,5884457,951798,...,0,0,0,0,0,0,0,5351828,27.5,male
SRR1555211,48027788,0,89024445,1348907,24399,1,148387777,3685378,113719675,2233983,...,0,0,0,1956002,0,0,0,0,27.5,male
SRR1555212,84899250,0,167383416,407713,37284600,1,219777948,58357903,79405469,147215871,...,0,1537700,0,824055,0,1040940,12712840,5952181,27.5,male


In [8]:
adata.loc[adata['Sex'].isna(), 'Sex'] = 'male'

In [9]:
adata.sort_values(by='Sex')

,TSPAN6,TNMD,DPM1,SCYL3,FIRRM,FGR,CFH,FUCA2,GCLC,NFYA,...,TUSC2P1,Unnamed: 34321,OR4M2-OT1,H2BK1,OR1Q1BP,Unnamed: 34325,Unnamed: 34326,TBCEL-TECTA,Age,Sex
Sample,,,,,,,,,,,,,,,,,,,,,
SRR13388751,222127430,406774,51853664,228028834,627791700,129589716,234481415,308033853,193496385,147703433,...,0,8000000,8097950,0,6663300,0,589174,13519400,80.0,female
SRR1555186,54149336,450349,55709971,264208,19333528,1,134109630,28594064,38540911,30889758,...,0,0,0,0,0,0,0,2263800,26.4,female
SRR13388736,253613257,198504,227481652,10651308,182165549,1,372074493,303323305,207882783,37430378,...,0,5000000,26685800,3346645,0,0,1889847,15203500,35.0,female
SRR1555180,28155151,0,10923412,8565773,869736,9817,85508014,34630306,41410065,24483270,...,0,0,0,0,0,0,0,0,26.4,female
SRR12021930,1121508954,1,2815570925,4300,99031446,1,3537198789,726219764,1946342166,5605136489,...,0,111000000,51652600,562889,48393800,0,314106732,471759000,83.0,female
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SRR8882187,260348025,6617,449953976,286,59320409,1,572959824,228953407,352765191,185557313,...,0,11000000,0,4797133,5000000,0,14879,19538180,67.0,male
SRR8882189,153466558,0,324063117,1,17620566,1,837300176,98261542,288671156,48668484,...,0,2000000,4000000,0,0,0,138749,3794310,72.0,male
SRR8882191,96204612,0,293431233,123715,15723598,1,499703693,108312058,245043098,228516317,...,0,0,1698520,3495605,0,2151350,193576,20604300,81.0,male


In [10]:
#----------------------------------------

In [10]:
age

'young'

In [11]:
adata

,TSPAN6,TNMD,DPM1,SCYL3,FIRRM,FGR,CFH,FUCA2,GCLC,NFYA,...,TUSC2P1,Unnamed: 34321,OR4M2-OT1,H2BK1,OR1Q1BP,Unnamed: 34325,Unnamed: 34326,TBCEL-TECTA,Age,Sex
Sample,,,,,,,,,,,,,,,,,,,,,
SRR13758984,21281762,0,15759712,0,11295483,0,35294310,0,18333093,0,...,0,0,0,0,0,0,2773,0,91.0,male
SRR13758985,8542646,0,15035432,0,0,0,26637602,0,1389125,15716930,...,0,0,0,0,0,0,1328,0,86.0,male
SRR13758986,4079945,0,5410942,0,5658191,0,10946301,4491317,7747412,9046855,...,0,0,0,0,0,0,1,0,69.0,male
SRR13758987,14758557,0,12809471,0,0,0,24093106,0,11562944,16086208,...,0,0,0,0,0,0,1614,0,83.0,male
SRR13758988,3623061,0,15529838,23957067,0,0,25492722,10773730,12889518,0,...,0,0,0,0,0,0,795,0,71.0,male
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SRR1555210,37402439,0,21355420,1,1212397,1,87213838,3370216,5884457,951798,...,0,0,0,0,0,0,0,5351828,27.5,male
SRR1555211,48027788,0,89024445,1348907,24399,1,148387777,3685378,113719675,2233983,...,0,0,0,1956002,0,0,0,0,27.5,male
SRR1555212,84899250,0,167383416,407713,37284600,1,219777948,58357903,79405469,147215871,...,0,1537700,0,824055,0,1040940,12712840,5952181,27.5,male


In [11]:
age_series = adata['Age']
age_series = age_series.apply(lambda age: 'young' if age < 35 else ('old' if age >= 65 else 'middle'))
adata['Age'] = age_series


In [12]:
adata = adata[adata['Age']==age]
adata.head(2)

,TSPAN6,TNMD,DPM1,SCYL3,FIRRM,FGR,CFH,FUCA2,GCLC,NFYA,...,TUSC2P1,Unnamed: 34321,OR4M2-OT1,H2BK1,OR1Q1BP,Unnamed: 34325,Unnamed: 34326,TBCEL-TECTA,Age,Sex
Sample,,,,,,,,,,,,,,,,,,,,,
SRR13759007,9058133,0,13819066,0,0,0,8772757,0,11309340,0,...,0,0,0,0,0,0,4,0,young,male
SRR13759008,10163774,0,12277757,0,0,0,22561889,7683369,873349,15315836,...,0,0,0,0,0,0,0,0,young,male


In [13]:
adata.sort_values(by='Sex')

,TSPAN6,TNMD,DPM1,SCYL3,FIRRM,FGR,CFH,FUCA2,GCLC,NFYA,...,TUSC2P1,Unnamed: 34321,OR4M2-OT1,H2BK1,OR1Q1BP,Unnamed: 34325,Unnamed: 34326,TBCEL-TECTA,Age,Sex
Sample,,,,,,,,,,,,,,,,,,,,,
SRR8882196,344034025,90410,454243692,12225,16161517,6,1051727220,72112363,408316427,14310379,...,0,2000000,5415790,0,0,0,6,20877400,young,female
SRR1555187,59635244,440532,57166422,740972,37962258,1,133820350,100120585,18499096,6259179,...,0,0,0,923021,0,0,0,0,young,female
SRR13388756,308309646,0,433643625,22671671,11167149,1,1418353093,86271415,492190148,85845023,...,0,0,0,0,0,0,153890,27184100,young,female
SRR13388757,421521918,128990,649589642,9735,21871683,7,1268289232,108561544,533675116,43283213,...,0,2000000,5415790,0,0,0,39,20877400,young,female
SRR13388758,238740709,0,426956390,9,9826608,6,733613338,120915432,205600186,15226949,...,0,3000000,0,0,0,0,3342925,7448610,young,female
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
SRR13388733,246390090,0,429272223,223547514,2969676,1,853923377,136323259,118059207,424269340,...,0,0,0,0,0,0,93771031,34130540,young,male
SRR13388732,179856618,0,293556065,2,46629247,30,437164137,266482924,242216546,43542420,...,3000000,2000000,13375200,0,0,0,20203430,7620840,young,male
SRR8882200,268157858,0,77028448,983,42403412,695,304378693,86302077,22832854,4239,...,4000000,2000000,0,0,3414570,0,401484,18755200,young,male


In [ ]:
#adata=adata.astype(int)
#adata.isnull().sum().any()

In [19]:
adata=adata.drop('Age',axis=1)
sex= adata['Sex']
adata=adata.drop('Sex',axis=1)

In [20]:

andata=AnnData(adata.to_numpy(), obs=pd.DataFrame(sex))
andata.obs_names = adata.index
andata.var_names = adata.columns


In [21]:
#adata = andata #AnnData(adata.to_numpy(), dtype=np.int32)
andata.var_names_make_unique()
andata

AnnData object with n_obs × n_vars = 91 × 34327
    obs: 'Sex'

In [22]:
# Process treatment information
#adata.obs['condition'] = age_series.apply(lambda age: 'young' if age < 35 else ('old' if age >= 65 else 'middle'))
andata.obs['condition'] = sex

In [23]:
andata.obs['condition']

Sample
SRR13759007    male
SRR13759008    male
SRR13759009    male
SRR13759010    male
SRR13759011    male
               ... 
SRR1555210     male
SRR1555211     male
SRR1555212     male
SRR1555213     male
SRR1555214     male
Name: condition, Length: 91, dtype: object

In [24]:
# Obtain genes that pass the thresholds
genes = dc.filter_by_expr(andata, group='condition', min_count=10, min_total_count=15, large_n=1, min_prop=1)

# Filter by these genes
andata = andata[:, genes].copy()
andata

AnnData object with n_obs × n_vars = 91 × 17977
    obs: 'Sex', 'condition'

In [26]:
# Build DESeq2 object
inference = DefaultInference(n_cpus=8)
dds = DeseqDataSet(
    adata=andata,
    design_factors='condition',
    refit_cooks=True,
    inference=inference,
)


In [27]:
dds.deseq2()

Fitting size factors...
... done in 0.03 seconds.

Fitting dispersions...
... done in 9.94 seconds.

Fitting dispersion trend curve...
... done in 0.29 seconds.

Fitting MAP dispersions...
... done in 16.19 seconds.

Fitting LFCs...
... done in 14.91 seconds.

Calculating cook's distance...
... done in 0.14 seconds.

Replacing 3688 outlier genes.

Fitting dispersions...
... done in 3.10 seconds.

Fitting MAP dispersions...
... done in 2.00 seconds.

Fitting LFCs...
... done in 3.76 seconds.



In [28]:
def get_DDS(younger_group, older_group, sex,  save=True):
    comparison = f'{younger_group}.vs.{older_group}'
    stat_res = DeseqStats(
        dds,
        contrast=["condition", younger_group, older_group],
        inference=inference
    )
    stat_res.summary()
    results_df = stat_res.results_df
    if not results_df is None:
        results_df.to_csv(f'/home/amore/work/data/{experiment}_{comparison}_{sex}_DDS.csv', header=True)
    return results_df

In [30]:
get_DDS(younger_group="male", older_group="female", sex="Young")

Running Wald tests...
... done in 0.98 seconds.



Log2 fold change & Wald test p-value: condition male vs female
                    baseMean  log2FoldChange     lfcSE      stat    pvalue  \
TSPAN6          5.772846e+07       -0.157837  0.137353 -1.149133  0.250501   
DPM1            7.926595e+07        0.237449  0.349514  0.679368  0.496905   
SCYL3           9.199393e+06        5.250315  1.204117  4.360302  0.000013   
FIRRM           2.021332e+07        0.878995  0.786325  1.117853  0.263630   
FGR             8.955568e+04       -0.577497  1.351998 -0.427144  0.669275   
...                      ...             ...       ...       ...       ...   
C4orf36.1       4.279024e+07        0.213917  0.373971  0.572016  0.567311   
Unnamed: 34321  4.431327e+05       -0.100227  2.030725 -0.049355  0.960636   
H2BK1           5.306162e+05        0.290439  1.862072  0.155976  0.876052   
Unnamed: 34326  3.081523e+06        3.857276  1.570359  2.456302  0.014038   
TBCEL-TECTA     2.618072e+06       -0.142345  1.081394 -0.131631  0.895276   



,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
TSPAN6,5.772846e+07,-0.157837,0.137353,-1.149133,0.250501,0.756342
DPM1,7.926595e+07,0.237449,0.349514,0.679368,0.496905,0.932802
SCYL3,9.199393e+06,5.250315,1.204117,4.360302,0.000013,0.000546
FIRRM,2.021332e+07,0.878995,0.786325,1.117853,0.263630,0.773280
FGR,8.955568e+04,-0.577497,1.351998,-0.427144,0.669275,0.977369
...,...,...,...,...,...,...
C4orf36.1,4.279024e+07,0.213917,0.373971,0.572016,0.567311,0.957891
Unnamed: 34321,4.431327e+05,-0.100227,2.030725,-0.049355,0.960636,0.997925
H2BK1,5.306162e+05,0.290439,1.862072,0.155976,0.876052,0.996621
Unnamed: 34326,3.081523e+06,3.857276,1.570359,2.456302,0.014038,0.141058


In [31]:
get_DDS(younger_group="male", older_group="female", sex="Old")

Running Wald tests...
... done in 1.01 seconds.



Log2 fold change & Wald test p-value: condition male vs female
                    baseMean  log2FoldChange     lfcSE      stat    pvalue  \
TSPAN6          5.772846e+07       -0.157837  0.137353 -1.149133  0.250501   
DPM1            7.926595e+07        0.237449  0.349514  0.679368  0.496905   
SCYL3           9.199393e+06        5.250315  1.204117  4.360302  0.000013   
FIRRM           2.021332e+07        0.878995  0.786325  1.117853  0.263630   
FGR             8.955568e+04       -0.577497  1.351998 -0.427144  0.669275   
...                      ...             ...       ...       ...       ...   
C4orf36.1       4.279024e+07        0.213917  0.373971  0.572016  0.567311   
Unnamed: 34321  4.431327e+05       -0.100227  2.030725 -0.049355  0.960636   
H2BK1           5.306162e+05        0.290439  1.862072  0.155976  0.876052   
Unnamed: 34326  3.081523e+06        3.857276  1.570359  2.456302  0.014038   
TBCEL-TECTA     2.618072e+06       -0.142345  1.081394 -0.131631  0.895276   



,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
TSPAN6,5.772846e+07,-0.157837,0.137353,-1.149133,0.250501,0.756342
DPM1,7.926595e+07,0.237449,0.349514,0.679368,0.496905,0.932802
SCYL3,9.199393e+06,5.250315,1.204117,4.360302,0.000013,0.000546
FIRRM,2.021332e+07,0.878995,0.786325,1.117853,0.263630,0.773280
FGR,8.955568e+04,-0.577497,1.351998,-0.427144,0.669275,0.977369
...,...,...,...,...,...,...
C4orf36.1,4.279024e+07,0.213917,0.373971,0.572016,0.567311,0.957891
Unnamed: 34321,4.431327e+05,-0.100227,2.030725,-0.049355,0.960636,0.997925
H2BK1,5.306162e+05,0.290439,1.862072,0.155976,0.876052,0.996621
Unnamed: 34326,3.081523e+06,3.857276,1.570359,2.456302,0.014038,0.141058


In [32]:
get_DDS(younger_group="male", older_group="female", sex="Middle")

Running Wald tests...
... done in 0.98 seconds.



Log2 fold change & Wald test p-value: condition male vs female
                    baseMean  log2FoldChange     lfcSE      stat    pvalue  \
TSPAN6          5.772846e+07       -0.157837  0.137353 -1.149133  0.250501   
DPM1            7.926595e+07        0.237449  0.349514  0.679368  0.496905   
SCYL3           9.199393e+06        5.250315  1.204117  4.360302  0.000013   
FIRRM           2.021332e+07        0.878995  0.786325  1.117853  0.263630   
FGR             8.955568e+04       -0.577497  1.351998 -0.427144  0.669275   
...                      ...             ...       ...       ...       ...   
C4orf36.1       4.279024e+07        0.213917  0.373971  0.572016  0.567311   
Unnamed: 34321  4.431327e+05       -0.100227  2.030725 -0.049355  0.960636   
H2BK1           5.306162e+05        0.290439  1.862072  0.155976  0.876052   
Unnamed: 34326  3.081523e+06        3.857276  1.570359  2.456302  0.014038   
TBCEL-TECTA     2.618072e+06       -0.142345  1.081394 -0.131631  0.895276   



,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
TSPAN6,5.772846e+07,-0.157837,0.137353,-1.149133,0.250501,0.756342
DPM1,7.926595e+07,0.237449,0.349514,0.679368,0.496905,0.932802
SCYL3,9.199393e+06,5.250315,1.204117,4.360302,0.000013,0.000546
FIRRM,2.021332e+07,0.878995,0.786325,1.117853,0.263630,0.773280
FGR,8.955568e+04,-0.577497,1.351998,-0.427144,0.669275,0.977369
...,...,...,...,...,...,...
C4orf36.1,4.279024e+07,0.213917,0.373971,0.572016,0.567311,0.957891
Unnamed: 34321,4.431327e+05,-0.100227,2.030725,-0.049355,0.960636,0.997925
H2BK1,5.306162e+05,0.290439,1.862072,0.155976,0.876052,0.996621
Unnamed: 34326,3.081523e+06,3.857276,1.570359,2.456302,0.014038,0.141058
